# Modèle ML — Détection de Fraude Bancaire
**AI Banking Risk Assistant**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

df = pd.read_csv('../data/creditcard.csv')
print('Shape:', df.shape)
df.head()

## 1. Prétraitement

In [ ]:
# Normaliser Amount et Time
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time'] = scaler.fit_transform(df[['Time']])

X = df.drop(columns=['Class'])
y = df['Class']

print('Distribution avant SMOTE:')
print(y.value_counts())

## 2. SMOTE — Rééquilibrage des classes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Distribution après SMOTE:')
print(pd.Series(y_train_sm).value_counts())

## 3. Entraînement XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train_sm, y_train_sm)
print('Modèle entraîné ')

## 4. Évaluation du modèle

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print('--- Rapport de classification ---')
print(classification_report(y_test, y_pred))
print('AUC-ROC:', roc_auc_score(y_test, y_proba))

# Matrice de confusion
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion')
plt.savefig('../reports/confusion_matrix.png')
plt.show()

## 5. Sauvegarde du modèle

In [ ]:
joblib.dump(model, '../models/fraud_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('Modèle sauvegardé ')